# Nexar Anticipation — آموزش مدل هشدار زودهنگام (Training v1)

<div dir="rtl">

این نوت‌بوک بعد از نوت‌بوک آماده‌سازی (`Nexar_Anticipation_Preprocess_FINAL_Colab.ipynb`) اجرا می‌شود و از همان پوشهٔ Drive می‌خواند. چیزی دانلود یا دوباره استخراج نمی‌شود.

**مدل:** `DINOv2-small` (ثابت، از مرحلهٔ قبل) + توجه روی ۵ توکن هر فریم + `GRU` علّی (فقط گذشته را می‌بیند) + سه خروجی:

1. خطر تصادف در ۱٫۵ ثانیهٔ آینده (دو‌حالته)
2. احتمال تصادف تا ۰٫۵ / ۱ / ۱٫۵ ثانیه (سه رگرسیون لجستیک پشت هم، مثل ایدهٔ «ستون AI»)
3. زمان باقی‌مانده تا تصادف (`time_to_accident`) بین ۰ تا ۱٫۵ ثانیه

بعد از آموزش: مقایسه با `time_of_alert`، تخمین لحظهٔ حادثه، جدول اعتمادپذیری در شب/روز/باران/صحنه، آزمایش کمترین فریم و طول پنجره، و توضیح‌پذیری (کدام فریم و کدام ربع تصویر مهم بوده).
</div>

In [ ]:
import copy, json, math, random, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/Nexar_Anticipation_FINAL")   # same folder as the preprocessing notebook
DATA_DIR, FEATURE_DIR = DRIVE_DIR / "data", DRIVE_DIR / "features"
OUT_DIR = DRIVE_DIR / "model_outputs"; OUT_DIR.mkdir(exist_ok=True)

HORIZON_S = 1.5
HORIZONS = (0.5, 1.0, 1.5)
MAX_T = 24                  # a 2 s window at 10 FPS has about 21 frames
FAR_TARGET = 0.20           # allowed false-alarm rate on normal videos (used to choose the alert threshold on val only)
MAX_LEAD_S = 3.0            # an alert earlier than this before the crash is NOT counted as a correct warning
K_EST = 3                   # event time = median of the first K alert windows (end time + predicted time-to-accident)
SEEDS = (0, 1, 2)
EPOCHS, BATCH, LR = 40, 64, 1e-3
AUG_STRIDES = [1, 1, 1, 2, 5, 10]        # random frame skipping during training -> robust to low FPS
AUG_SECONDS = [None, None, 0.5, 1.0, 1.5]  # random shorter windows during training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
print("device:", device)

## ۱ — خواندن داده و ابزارهای پنجره
<div dir="rtl">فقط فایل‌هایی که نوت‌بوک قبلی ساخته خوانده می‌شود. ویژگی‌ها از فایل‌های NPZ بخش‌به‌بخش در حافظه می‌آیند.</div>

In [ ]:
train_w = pd.read_csv(DATA_DIR / "train_windows.csv")
val_w   = pd.read_csv(DATA_DIR / "val_windows.csv")
test_w  = pd.read_csv(DATA_DIR / "test_internal_windows.csv")
videos_out = pd.read_csv(DATA_DIR / "videos_with_features.csv")
print({k: len(v) for k, v in {"train": train_w, "val": val_w, "test_internal": test_w}.items()})

def window_batches(df, max_windows=2000):
    """Yield (row_positions, X[N,MAX_T,5,384] float16, T[N,MAX_T], L[N]) one feature file at a time (right padded)."""
    assert (df.index == np.arange(len(df))).all(), "reset_index(drop=True) first"
    for fname, grp in df.groupby("feature_file", sort=False):
        with np.load(FEATURE_DIR / fname, allow_pickle=False) as z:
            feats, times = z["features"], z["times"]
        for s in range(0, len(grp), max_windows):
            sub = grp.iloc[s:s + max_windows]
            n = len(sub)
            X = np.zeros((n, MAX_T, 5, 384), np.float16); T = np.zeros((n, MAX_T), np.float32); L = np.zeros(n, np.int64)
            for i, (a, b) in enumerate(zip(sub.index_start.to_numpy(), sub.index_end.to_numpy())):
                a = max(int(a), int(b) - MAX_T); b = int(b)
                X[i, :b - a] = feats[a:b]; T[i, :b - a] = times[a:b]; L[i] = b - a
            yield sub.index.to_numpy(), X, T, L

def build_tensors(df):
    N = len(df)
    X = np.zeros((N, MAX_T, 5, 384), np.float16); T = np.zeros((N, MAX_T), np.float32); L = np.zeros(N, np.int64)
    for pos, x, t, l in window_batches(df):
        X[pos], T[pos], L[pos] = x, t, l
    return X, T, L

def make_targets(df):
    risk = df.risk_target.to_numpy().astype(np.float32)
    tta = df.tta_target_s.fillna(0).to_numpy().astype(np.float32)
    mask = df.tta_mask.to_numpy().astype(np.float32)
    hor = np.stack([((risk == 1) & (mask == 1) & (tta <= h + 1e-6)) for h in HORIZONS], 1).astype(np.float32)
    return risk, tta, mask, hor

def subsample(x, t, l, stride=1, last_seconds=None):
    """Keep the LAST frame, then every `stride`-th frame going back; optionally only the last `last_seconds`."""
    idx = np.arange(l - 1, -1, -stride)[::-1]
    if last_seconds is not None:
        idx = idx[t[idx] >= t[l - 1] - last_seconds - 1e-4]
    n = len(idx)
    xo = np.zeros_like(x); to = np.zeros_like(t)
    xo[:n] = x[idx]; to[:n] = t[idx]
    return xo, to, n

def make_batch(items):
    x = torch.from_numpy(np.stack([a for a, _, _ in items]).astype(np.float32))
    t = np.stack([b for _, b, _ in items]); l = np.array([c for _, _, c in items])
    dt = np.zeros_like(t); dt[:, 1:] = np.clip(np.diff(t, axis=1), 0, 1)
    dt = dt * (np.arange(t.shape[1])[None, :] < l[:, None])
    return x, torch.from_numpy(dt.astype(np.float32)), torch.from_numpy(l.astype(np.int64))

class WinDS(Dataset):
    def __init__(self, arrays, targets, train=False):
        self.X, self.T, self.L = arrays; self.y = targets; self.train = train
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x, t, l = self.X[i], self.T[i], int(self.L[i])
        if self.train:
            x, t, l = subsample(x, t, l, random.choice(AUG_STRIDES), random.choice(AUG_SECONDS))
        return (x, t, l), [yy[i] for yy in self.y]

def collate(batch):
    x, dt, l = make_batch([b[0] for b in batch])
    ys = [torch.tensor(np.array([b[1][k] for b in batch])).float() for k in range(4)]
    return (x, dt, l, *ys)

t0 = time.time()
arr_train, arr_val, arr_test = build_tensors(train_w), build_tensors(val_w), build_tensors(test_w)
tg_train, tg_val, tg_test = make_targets(train_w), make_targets(val_w), make_targets(test_w)
print("tensors built in", round(time.time() - t0), "s | train X:", arr_train[0].shape)

## ۲ — مدل پایه: لجستیک رگرسیون
<div dir="rtl">یک مدل ساده و بدون شبکهٔ عصبی، فقط برای مقایسه. اگر مدل اصلی از این بهتر نباشد، پیچیدگی اضافه ارزشی ندارد. سه رگرسیون لجستیک برای «تصادف تا ۰٫۵ / ۱ / ۱٫۵ ثانیه» هم همین‌جا آموزش داده می‌شود.</div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def lr_features(arrays):
    X, T, L = arrays
    out = np.zeros((len(X), 2 * 5 * 384), np.float32)
    for i in range(len(X)):
        x = X[i, :L[i]].astype(np.float32)
        out[i] = np.concatenate([x.mean(0).ravel(), (x[-1] - x[0]).ravel()])
    return out

Ftr, Fva = lr_features(arr_train), lr_features(arr_val)
y_tr_risk, y_va_risk = tg_train[0], tg_val[0]
base = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000)).fit(Ftr, y_tr_risk)
p = base.predict_proba(Fva)[:, 1]
print(f"LogReg baseline | risk AUC {roc_auc_score(y_va_risk, p):.3f} | AP {average_precision_score(y_va_risk, p):.3f}")
for j, h in enumerate(HORIZONS):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000)).fit(Ftr, tg_train[3][:, j])
    print(f"  logistic 'crash within {h}s' | val AUC {roc_auc_score(tg_val[3][:, j], m.predict_proba(Fva)[:, 1]):.3f}")

## ۳ — مدل اصلی
<div dir="rtl">

- هر فریم پنج بردار دارد (کل تصویر + چهار ربع). مدل یاد می‌گیرد به کدام یک بیشتر توجه کند. وزن‌های توجه بعداً برای توضیح‌پذیری استفاده می‌شوند.
- حرکت با تفاضل فریم‌های پشت‌سرهم و فاصلهٔ زمانی بین فریم‌ها وارد می‌شود، پس مدل با ۱۰ یا ۵ یا ۲ فریم بر ثانیه هم کار می‌کند.
- `GRU` فقط گذشته را می‌بیند، مثل سیستم واقعی داخل خودرو.
- حین آموزش فریم‌ها تصادفی کم می‌شوند و پنجره گاهی کوتاه می‌شود تا مدل به نرخ فریم و طول پنجره وابسته نشود.
- سه مدل با seedهای مختلف آموزش داده و میانگین گرفته می‌شود تا نتیجه پایدارتر باشد.
</div>

In [ ]:
class AnticipationNet(nn.Module):
    def __init__(self, d_in=384, d=192, hidden=128, layers=2, p=0.3, tok_drop=0.15):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.embed = nn.Linear(d_in, d)
        self.tok_id = nn.Parameter(torch.randn(5, d) * 0.02)       # which of the 5 tokens: whole image, TL, TR, BL, BR
        self.score = nn.Sequential(nn.Linear(d, 64), nn.Tanh(), nn.Linear(64, 1))
        self.drop = nn.Dropout(p); self.tok_drop = tok_drop
        self.gru = nn.GRU(2 * d + 1, hidden, layers, batch_first=True, dropout=p)
        self.head = nn.Sequential(nn.Linear(hidden, 64), nn.GELU(), nn.Dropout(p), nn.Linear(64, 1 + len(HORIZONS) + 1))

    def forward(self, x, dt, lengths):
        h = self.embed(self.norm(x)) + self.tok_id                   # B,T,5,d
        sc = self.score(h).squeeze(-1)                               # B,T,5
        if self.training and self.tok_drop > 0:
            drop = torch.rand_like(sc) < self.tok_drop; drop[..., 0] = False
            sc = sc.masked_fill(drop, -1e4)
        attn = torch.softmax(sc, -1)
        f = self.drop((attn.unsqueeze(-1) * h).sum(2))               # B,T,d
        delta = torch.zeros_like(f); delta[:, 1:] = f[:, 1:] - f[:, :-1]
        inp = torch.cat([f, delta, dt.unsqueeze(-1)], -1)
        packed = pack_padded_sequence(inp, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hn = self.gru(packed)
        o = self.head(hn[-1])
        return {"risk": o[:, 0], "hor": o[:, 1:1 + len(HORIZONS)], "tta": HORIZON_S * torch.sigmoid(o[:, -1]), "attn": attn}

def compute_loss(o, risk, tta, mask, hor, pos_weight):
    l_risk = F.binary_cross_entropy_with_logits(o["risk"], risk, pos_weight=pos_weight)
    l_hor = F.binary_cross_entropy_with_logits(o["hor"], hor)
    l_tta = (F.huber_loss(o["tta"], tta, reduction="none", delta=0.2) * mask).sum() / mask.sum().clamp(min=1)
    return l_risk + 0.5 * l_hor + 2.0 * l_tta

@torch.no_grad()
def predict_arrays(models, X, T, L, stride=1, last_seconds=None, batch=256):
    for m in models: m.eval()
    N = len(X)
    risk = np.zeros(N, np.float32); hor = np.zeros((N, len(HORIZONS)), np.float32); tta = np.zeros(N, np.float32)
    for b in range(0, N, batch):
        e = min(N, b + batch)
        x, dt, l = make_batch([subsample(X[i], T[i], int(L[i]), stride, last_seconds) for i in range(b, e)])
        x, dt = x.to(device), dt.to(device)
        for m in models:
            o = m(x, dt, l)
            risk[b:e] += torch.sigmoid(o["risk"]).cpu().numpy() / len(models)
            hor[b:e] += torch.sigmoid(o["hor"]).cpu().numpy() / len(models)
            tta[b:e] += o["tta"].cpu().numpy() / len(models)
    return {"risk": risk, "hor": hor, "tta": tta}

def predict_df(models, df, **kw):
    N = len(df)
    out = {"risk": np.zeros(N, np.float32), "hor": np.zeros((N, len(HORIZONS)), np.float32), "tta": np.zeros(N, np.float32)}
    for pos, X, T, L in window_batches(df):
        p = predict_arrays(models, X, T, L, **kw)
        for k in out: out[k][pos] = p[k]
    return out

def window_metrics(df, pred):
    y = df.risk_target.to_numpy(); m = df.tta_mask.to_numpy() == 1
    tta_true = df.tta_target_s.to_numpy()
    res = {"risk_AUC": roc_auc_score(y, pred["risk"]), "risk_AP": average_precision_score(y, pred["risk"]),
           "tta_MAE_s": float(np.abs(pred["tta"][m] - tta_true[m]).mean())}
    for j, h in enumerate(HORIZONS):
        lab = m & (np.nan_to_num(tta_true, nan=9.0) <= h + 1e-6)
        res[f"AUC_within_{h}s"] = roc_auc_score(lab, pred["hor"][:, j])
    return res

tta_mean_train = float(train_w.tta_target_s[train_w.tta_mask == 1].mean())
mv = val_w.tta_mask.to_numpy() == 1
print("Reference: always guessing the mean time-to-accident gives MAE =",
      round(float(np.abs(val_w.tta_target_s.to_numpy()[mv] - tta_mean_train).mean()), 3), "s (the model must beat this)")

## ۴ — آموزش
<div dir="rtl">بهترین epoch با امتیاز `AUC - 0.5 × MAE` روی Validation انتخاب می‌شود (هم تشخیص و هم دقت زمان). Test داخلی اینجا استفاده نمی‌شود.</div>

In [ ]:
def train_one(seed):
    set_seed(seed)
    model = AnticipationNet().to(device)
    ds = WinDS(arr_train, tg_train, train=True)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl))
    n_pos = float(tg_train[0].sum()); pos_weight = torch.tensor((len(tg_train[0]) - n_pos) / n_pos, device=device)
    best, best_state, bad = -9.0, None, 0
    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        for x, dt, l, risk, tta, mask, hor in dl:
            x, dt, risk, tta, mask, hor = [a.to(device) for a in (x, dt, risk, tta, mask, hor)]
            loss = compute_loss(model(x, dt, l), risk, tta, mask, hor, pos_weight)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item()
        m = window_metrics(val_w, predict_arrays([model], *arr_val))
        score = m["risk_AUC"] - 0.5 * m["tta_MAE_s"]
        if score > best: best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        if ep % 3 == 0 or bad == 0:
            print(f"seed {seed} ep {ep:02d} | loss {tot/len(dl):.3f} | val AUC {m['risk_AUC']:.3f} AP {m['risk_AP']:.3f} MAE {m['tta_MAE_s']:.3f}s | score {score:.3f}{' *' if bad == 0 else ''}")
        if bad >= 8: break
    model.load_state_dict(best_state)
    torch.save(best_state, OUT_DIR / f"anticipation_gru_seed{seed}.pt")
    return model

t0 = time.time()
models = [train_one(s) for s in SEEDS]
print("training done in", round((time.time() - t0) / 60, 1), "min")
vm = window_metrics(val_w, predict_arrays(models, *arr_val))
print("ENSEMBLE on val windows:", {k: round(float(v), 3) for k, v in vm.items()})

## ۵ — هشدار روی ویدیوی پیوسته و مقایسه با `time_of_alert`
<div dir="rtl">

مدل روی ویدیو هر ۰٫۵ ثانیه یک عدد خطر می‌دهد. **آستانهٔ هشدار فقط روی Validation انتخاب می‌شود**: بیشترین درصد هشدار به‌موقع، به شرط اینکه روی ویدیوهای عادی از `FAR_TARGET` بیشتر هشدار اشتباه ندهد.

- **هشدار به‌موقع:** اولین هشدار بعد از شروع و قبل از تصادف، و حداکثر `MAX_LEAD_S` ثانیه زودتر.
- **زودتر از انسان:** `time_of_alert - زمان هشدار مدل`. مثبت یعنی مدل زودتر از برچسب‌گذار هشدار داده است.
- **تخمین لحظهٔ حادثه:** زمان انتهای پنجره + `time_to_accident` پیش‌بینی‌شده، میانهٔ اولین چند هشدار.
</div>

In [ ]:
vid_info = videos_out[["video_path", "label"]]
def load_timeline(prefix):
    w = pd.read_csv(DATA_DIR / f"{prefix}_timeline_windows.csv")
    m = pd.read_csv(DATA_DIR / f"{prefix}_timeline_eval_only.csv").drop(columns=["video_path"])
    w = w.merge(m, on="sample_id", validate="one_to_one").merge(vid_info, on="video_path", validate="many_to_one")
    return w.reset_index(drop=True)

def timeline_with_predictions(prefix):
    tl = load_timeline(prefix)
    pr = predict_df(models, tl)
    tl["p"], tl["tta_pred"] = pr["risk"], pr["tta"]
    return tl

def make_groups(tl):
    tl = tl.sort_values(["video_path", "window_end_time"])
    return [g for _, g in tl.groupby("video_path", sort=False)]

def per_video(groups, thr):
    rows = []
    for g in groups:
        hit = g[g.p >= thr]; r0 = g.iloc[0]; label = int(r0.label)
        r = {"video_path": r0.video_path, "label": label, "alert": len(hit) > 0, "weather": r0.weather,
             "scene": r0.scene, "light_conditions": r0.light_conditions,
             "first_alert_time": np.nan, "lead_s": np.nan, "earlier_than_human_s": np.nan, "event_err_s": np.nan}
        if len(hit):
            ta = float(hit.window_end_time.iloc[0]); r["first_alert_time"] = ta
            if label == 1:
                ev, ha = float(r0.time_of_event), float(r0.time_of_alert)
                r["lead_s"] = ev - ta; r["earlier_than_human_s"] = ha - ta
                est = float((hit.window_end_time + hit.tta_pred).iloc[:K_EST].median())
                r["event_err_s"] = est - ev
        rows.append(r)
    return pd.DataFrame(rows)

def summarize(pv):
    pos, neg = pv[pv.label == 1], pv[pv.label == 0]
    timely = pos.alert & (pos.lead_s > 0) & (pos.lead_s <= MAX_LEAD_S)
    return {"n_pos": len(pos), "n_neg": len(neg),
            "false_alarm_rate": float(neg.alert.mean()) if len(neg) else np.nan,
            "timely_rate": float(timely.mean()) if len(pos) else np.nan,
            "median_lead_s": float(pos.lead_s[timely].median()) if timely.any() else np.nan,
            "earlier_than_human_pct": float((pos.earlier_than_human_s[timely] > 0).mean()) if timely.any() else np.nan,
            "median_earlier_than_human_s": float(pos.earlier_than_human_s[timely].median()) if timely.any() else np.nan,
            "event_time_MAE_s": float(pos.event_err_s.abs().mean()) if pos.event_err_s.notna().any() else np.nan}

val_tl = timeline_with_predictions("val")
val_groups = make_groups(val_tl)
sweep = pd.DataFrame([{**summarize(per_video(val_groups, t)), "thr": t} for t in np.round(np.arange(0.05, 0.96, 0.01), 2)])
ok = sweep[sweep.false_alarm_rate <= FAR_TARGET]
best = (ok.sort_values(["timely_rate", "false_alarm_rate"], ascending=[False, True]).iloc[0] if len(ok)
        else sweep.sort_values("false_alarm_rate").iloc[0])
THR = float(best.thr)
print(f"Chosen alert threshold (from VAL only): {THR}")
print(pd.DataFrame([best]).round(3).T)

## ۶ — ارزیابی نهایی روی Test داخلی (فقط یک بار)
<div dir="rtl">آستانه از Validation آمده و دیگر تغییر نمی‌کند. بعد از دیدن این نتیجه، مدل یا آستانه را برای بهتر شدن عدد تست تغییر ندهید.</div>

In [ ]:
test_pred = predict_arrays(models, *arr_test)
test_win = window_metrics(test_w, test_pred)
print("TEST windows:", {k: round(float(v), 3) for k, v in test_win.items()})

test_tl = timeline_with_predictions("test_internal")
test_pv = per_video(make_groups(test_tl), THR)
test_sum = summarize(test_pv)
print("TEST continuous video (threshold fixed from val):")
print(pd.Series(test_sum).round(3))
test_pv.to_csv(OUT_DIR / "test_internal_per_video_alerts.csv", index=False)
json.dump({"threshold": THR, "window_metrics": test_win, "video_summary": test_sum},
          open(OUT_DIR / "final_test_internal_results.json", "w"), indent=2, default=float)

## ۷ — مدل در چه شرایطی قابل اعتماد است؟
<div dir="rtl">برای هر گروه (روز/شب، آب‌وهوا، صحنه) سه عدد: درصد هشدار به‌موقع، هشدار اشتباه روی ویدیوهای عادی، و خطای تخمین لحظهٔ حادثه. گروه‌هایی که کمتر از ۸ ویدیو دارند عدد قابل اعتمادی نمی‌دهند و نمایش داده نمی‌شوند. چون ویدیوهای Test داخلی کم‌اند، جدول را روی Validation و Test با هم می‌سازیم.</div>

In [ ]:
all_pv = pd.concat([per_video(val_groups, THR), test_pv], ignore_index=True)
for col in ["light_conditions", "weather", "scene"]:
    rows = []
    for val, g in all_pv.groupby(col):
        if len(g) < 8: continue
        s = summarize(g); rows.append({col: val, "videos": len(g), **{k: s[k] for k in ["timely_rate", "false_alarm_rate", "median_lead_s", "event_time_MAE_s"]}})
    print(f"\n=== by {col} ==="); print(pd.DataFrame(rows).round(2).to_string(index=False))

## ۸ — کمترین تعداد فریم و کمترین طول پنجره
<div dir="rtl">همان مدل آموزش‌دیده، روی Validation، با نرخ فریم و طول پنجرهٔ کمتر ارزیابی می‌شود. اولین ردیف که افت `risk_AUC` آن از بهترین حالت کمتر از ۰٫۰۲ باشد، ارزان‌ترین تنظیم قابل قبول است. **کمترین رزولوشن** با این نوت‌بوک سنجیده نمی‌شود، چون ویژگی‌ها از قبل با ۳۹۲×۲۲۴ استخراج شده‌اند (توضیح در پاسخ چت).</div>

In [ ]:
rows = []
for fps, stride in [(10, 1), (5, 2), (2, 5), (1, 10)]:
    for sec in [0.5, 1.0, 1.5, 2.0]:
        pr = predict_arrays(models, *arr_val, stride=stride, last_seconds=sec)
        m = window_metrics(val_w, pr)
        rows.append({"fps": fps, "window_s": sec, "frames": int(round(sec * fps)) + 1,
                     "risk_AUC": m["risk_AUC"], "tta_MAE_s": m["tta_MAE_s"]})
abl = pd.DataFrame(rows)
print(abl.round(3).to_string(index=False))
best_auc = abl.risk_AUC.max()
cheap = abl[abl.risk_AUC >= best_auc - 0.02].sort_values(["frames", "fps"]).iloc[0]
print(f"\nCheapest setting within 0.02 AUC of the best: {int(cheap.fps)} FPS, {cheap.window_s}s window, {int(cheap.frames)} frames")

## ۹ — توضیح‌پذیری: کدام فریم و کدام قسمت تصویر؟
<div dir="rtl">

**توجه:** Grad-CAM به نقشهٔ مکانی دقیق از لایهٔ کانولوشن نیاز دارد. اینجا هر فریم فقط پنج بردار دارد، پس نقشهٔ دقیق ساخته نمی‌شود. به‌جایش از `گرادیان × ورودی` استفاده می‌کنیم که نشان می‌دهد:

- **کدام فریم** (چند دهم ثانیه قبل از انتهای پنجره) بیشترین اثر را روی خطر داشته
- **کدام ربع تصویر** (چپ‌بالا، راست‌بالا، چپ‌پایین، راست‌پایین) مهم بوده

این نمودار لحظهٔ دقیق تصادف را نمی‌دهد. لحظهٔ حادثه از بخش ۵ می‌آید (زمان هشدار + `time_to_accident` پیش‌بینی‌شده). اینجا فقط می‌بینیم مدل بر چه اساسی هشدار داده است.
</div>

In [ ]:
def saliency_for_window(model, X, T, L):
    model.eval()
    x, dt, l = make_batch([(X, T, int(L))])
    x = x.to(device).requires_grad_(True); dt = dt.to(device)
    with torch.backends.cudnn.flags(enabled=False):
        out = model(x, dt, l)
        out["risk"].sum().backward()
    g = (x.grad * x).abs().sum(-1)[0, :int(L)].detach().cpu().numpy()      # (frames, 5)
    return g / (g.sum() + 1e-12)

def explain_video(tl, video_path, thr=THR):
    g = tl[tl.video_path == video_path].sort_values("window_end_time").reset_index(drop=True)
    hit = g[g.p >= thr]
    row = hit.iloc[[0]] if len(hit) else g.iloc[[len(g) - 1]]
    pos, X, T, L = next(window_batches(row.reset_index(drop=True)))
    s = saliency_for_window(models[0], X[0], T[0], L[0])
    t_rel = T[0, :L[0]] - T[0, L[0] - 1]
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8), gridspec_kw={"width_ratios": [2.2, 1.6, 0.9]})
    ax[0].plot(g.window_end_time, g.p, "-o", ms=3); ax[0].axhline(thr, color="r", ls="--", label="threshold")
    r0 = g.iloc[0]
    if pd.notna(r0.time_of_event): ax[0].axvline(r0.time_of_event, color="k", label="real crash")
    if pd.notna(r0.time_of_alert): ax[0].axvline(r0.time_of_alert, color="g", ls=":", label="human alert")
    if len(hit):
        est = float((hit.window_end_time + hit.tta_pred).iloc[:K_EST].median())
        ax[0].axvline(est, color="orange", ls="--", label="estimated crash")
    ax[0].set_xlabel("video time (s)"); ax[0].set_ylabel("risk"); ax[0].legend(fontsize=8); ax[0].set_title(Path(video_path).name)
    ax[1].bar(t_rel, s.sum(1), width=0.08); ax[1].set_xlabel("seconds before window end"); ax[1].set_title("frame importance")
    quad = s[:, 1:].sum(0).reshape(2, 2)
    im = ax[2].imshow(quad, cmap="Reds"); ax[2].set_xticks([]); ax[2].set_yticks([])
    ax[2].set_title(f"image quarters\n(whole-image token {s[:, 0].sum():.0%})", fontsize=9); plt.colorbar(im, ax=ax[2], fraction=0.05)
    plt.tight_layout(); plt.show()

pos_videos = val_tl[val_tl.label == 1].video_path.drop_duplicates().tolist()
for vp in pos_videos[:4]:
    explain_video(val_tl, vp)

## ۱۰ — خلاصه برای گزارش
<div dir="rtl">اگر در بخش ۶ مدل از مدل پایه بهتر بود و `risk_AUC` تست داخلی با Validation نزدیک بود، مدل قابل دفاع است. اگر فاصلهٔ زیادی بود، یعنی روی Validation بیش‌ازحد تنظیم شده است.</div>

In [ ]:
print("Saved to:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()): print(" ", p.name, round(p.stat().st_size / 1e3), "KB")